# Passage analysis: TEMPLATE (RS-04)

Copy this template into `$SEASTONES_NOTEBOOK_DIR` (outside git, decision D-PLAN-1) and work there.

Rules (PRD RE-1, RE-2, RE-6; LEGAL-REVIEW 2.3):
- Run on this machine only. Never upload the audio, any excerpt, or any image of it.
- Do not paste spectrogram or waveform images of a recording into any file in the repo.
- Every result is a **hypothesis** for the ledger (`research/ledger/H-NNN.md`), with alternatives and a confidence.
- A transcription made from this notebook is `origin: seastones-derived`, `status: draft-hidden`, and stays in the private folder.

In [ ]:
# 1. The guard: stops here unless the private folders are set up safely.
from research.tools import env

paths = env.check()
paths

In [ ]:
# 2. Choose the passage. Type the file name here, in your private copy only.
import librosa

from research.tools import analysis as A

RECORDING = paths.audio_dir / "CHANGE-ME"  # a file in the private audio folder
START_S, END_S = 0.0, 30.0  # the passage's time range
y, sr = librosa.load(RECORDING, sr=None, mono=True, offset=START_S, duration=END_S - START_S)

In [ ]:
# 3. RE-7 questions 1 and 2: pitch track, step size, update rate.
# Set the band around one voice's fundamental (or one clear partial).
track = A.pitch_track(y, sr, fmin=200, fmax=1200, frame_length=1024, hop_length=128)
steps = A.detect_steps(track)
print(
    f"step {steps.step_cents:.2f} cents, quantum {steps.quantum_cents:.2f} cents, "
    f"update rate {steps.update_hz:.1f} Hz, {len(steps.plateau_cents)} plateaus"
)
print(steps.notes)
for rng in (5.0, 10.0, 10.24):
    print(
        f"if the D/A spans {rng} V at 1 V/oct: {A.implied_dac_bits(steps.quantum_cents, rng):.1f} bits"
    )

In [ ]:
# 4. RE-7 question 3: the computer rhythm.
on = A.onsets(y, sr)
loop = A.loop_length(on, tolerance_s=0.015)
print(len(on), "onsets;", loop)

In [ ]:
# 5. RE-7 question 5: zipper steps in a sweep (choose the sweep's time range).
print(A.zipper(track))

## Ledger draft

Write the entry by hand into `research/ledger/H-NNN.md` and run
`python -m research.tools.ledger validate`. Fill: observed (the numbers above), inference,
alternatives (tracker artefacts, vibrato, tape speed, mixing), confidence, status `hypothesis`.